
# 02 Preprocessing


# Import Libraries

In [0]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import numpy as np
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector


# Config

In [0]:
PATH_RAW_DATASET = '../src/data/creditcard.csv'
PROJECT_NAME = "pork_credit_card_fraud_detection"
TBL_RAW_DATASET = f"ctl_training_dev.m7_dev.{PROJECT_NAME}_raw_dataset"
TBL_DATASET = f"ctl_training_dev.m7_dev.{PROJECT_NAME}_dataset"
EXP_NAME = f"/Workspace/Users/umaporpa@ais.co.th/cicd-lab/src/experiment/{PROJECT_NAME}"
MODEL_NAME = PROJECT_NAME
RUN_NAME = MODEL_NAME


# Preprocessing

In [0]:
df = spark.table(TBL_RAW_DATASET)

In [0]:
df = df.toPandas()

In [0]:
df.columns

In [0]:
df.drop(['Time'],inplace=True, axis =1)

In [0]:
df

In [0]:
df.shape

In [0]:
feature_names = df.iloc[:, :29].columns
target = df.iloc[:, -1:].columns


data_features = df[feature_names]
data_target = df[target]

In [0]:
std_transformer = Pipeline(
    steps=[
        ("Imputer", SimpleImputer(strategy="mean")),
        ("scaler_transformer", StandardScaler()),
    ]
).set_output(transform="pandas")

In [0]:
with open(f'{EXP_NAME}_preproc.pkl', 'wb') as file:
    pickle.dump(std_transformer, file)

In [0]:
data_feature_std = std_transformer.fit_transform(data_features)

In [0]:
fruad_data_std = pd.concat([data_feature_std,data_target], axis =1)

In [0]:
fruad_data_std

In [0]:
df_dataset = spark.createDataFrame(fruad_data_std)
df_dataset.display()


# Write Table

In [0]:
print(f"Write dataset table '{TBL_DATASET}'")
(
    df_dataset.write.format("delta")
    # .partitionBy(["partition_date", "partition_time"])
    .mode("overwrite")
    # .option("partitionOverwriteMode", "dynamic")
    # .option("overwriteSchema", "true")
    .saveAsTable(TBL_DATASET)
)